In [ ]:
import pandas as pd
import spatialdata
import sopa
import anndata
import pathlib as pl

import scanpy as sc
import squidpy as sq

import matplotlib.pyplot as plt
import seaborn as sns
import spatialdata as sd
import palettable

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

import os

from tqdm.notebook import tqdm

# All viz functions

In [ ]:
def calculate_cell_type_proportions(adata, obs_key="Cell_type", position_key="spatial", 
                                    method="radius", radius=50, n_neighbors=10):
    """
    Calculate the proportion of each cell type in the neighborhood of each cell.

    Parameters:
    - adata: AnnData object containing cell information.
    - obs_key: Key in `adata.obs` that contains cell type labels.
    - position_key: Key in `adata.obsm` that contains spatial coordinates.
    - method: 'radius' to define neighborhood by spatial distance, 'knn' for k-nearest neighbors.
    - radius: Radius for the neighborhood if using 'radius' method.
    - n_neighbors: Number of neighbors if using 'knn' method.
    
    Returns:
    - Updated adata object with new columns in `adata.obs` representing the proportion of each cell type.
    """
    if obs_key not in adata.obs.columns:
        raise ValueError(f"{obs_key} not found in adata.obs.")

    if position_key not in adata.obsm:
        raise ValueError(f"{position_key} not found in adata.obsm.")

    # Extract positions and cell types
    positions = adata.obsm[position_key]
    cell_types = adata.obs[obs_key].values

    # Get unique cell types
    unique_cell_types = np.unique(cell_types)

    # Initialize dataframe to store results
    proportions = pd.DataFrame(index=adata.obs.index, columns=unique_cell_types, data=0)

    if method == "radius":
        # Use NearestNeighbors with radius-based search
        nbrs = NearestNeighbors(radius=radius).fit(positions)
        distances, indices = nbrs.radius_neighbors(positions)

        for i, neighbors in enumerate(indices):
            if len(neighbors) > 0:
                neighbor_types = cell_types[neighbors]
                for cell_type in unique_cell_types:
                    proportions.loc[adata.obs.index[i], cell_type] = np.sum(neighbor_types == cell_type) / len(neighbor_types)

    elif method == "knn":
        # Use NearestNeighbors with k-nearest neighbors
        nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(positions)
        distances, indices = nbrs.kneighbors(positions)

        for i, neighbors in enumerate(indices):
            neighbor_types = cell_types[neighbors]
            for cell_type in unique_cell_types:
                proportions.loc[adata.obs.index[i], cell_type] = np.sum(neighbor_types == cell_type) / len(neighbor_types)

    else:
        raise ValueError("Invalid method. Use 'radius' or 'knn'.")

    # Add to adata.obs
    for cell_type in unique_cell_types:
        adata.obs[f"prop_{cell_type}"] = proportions[cell_type]

    return adata


In [ ]:
def plot_predicted_cell_types(adata, obs_key="Cell_type", position_key="spatial",
                                         xlim=None, ylim=None, color_mapping=None, 
                                         highlight_cell_types=None, s=2, highlight_size=50, show_legend=True,
                                         figsize=(12, 10), show_axes=True,
                                          linewidth=0.2, savedir=None, savename=""):
    """
    Plots the predicted cell types from adata.obs["Cell_type"] and highlights specific cell types.

    Parameters:
    - adata: anndata object
    - obs_key: Key to access cell type labels in obs (default: "Cell_type")
    - position_key: Key to access cell coordinates in adata (default: "spatial")
    - xlim: Tuple (xmin, xmax) to zoom in on a specific x-range (default: None, no zoom)
    - ylim: Tuple (ymin, ymax) to zoom in on a specific y-range (default: None, no zoom)
    - color_mapping: A dictionary mapping cell types to RGB colors (default: None)
    - highlight_cell_types: List of cell types to highlight (default: None, highlights none)
    - s: Size of the circles for non-highlighted cells (default: 2)
    - highlight_size: Size of the circles for highlighted cells (default: 50)
    - figsize: Size of the figure (default: (12, 10))
    - show_legend: Whether to display the legend
    - show_axes: Whether to show the x and y axis
    - linewidth: the size of the line around the circle
    - savedir: If not none, will save the plot in this directory
    - savename: This is used to create save the plot in savedir / savename / namefig.png
    
    """
    # Extract table and cell type labels
    cell_types = adata.obs[obs_key]  # Categorical cell types
    
    # Use provided color_mapping or generate one based on the cell types
    if color_mapping is None:
        # Generate a color mapping if not provided
        unique_cell_types = np.sort(cell_types.unique())  # Sort for consistent color mapping
        palette = sns.color_palette("tab20", len(unique_cell_types)) if len(unique_cell_types) <= 20 else sns.color_palette("husl", len(unique_cell_types))
        color_mapping = {cell_type: palette[i] for i, cell_type in enumerate(unique_cell_types)}

    # Extract spatial positions
    positions = adata.obsm[position_key]
    x, y = positions[:, 0], positions[:, 1]

    # Apply zoom filter if xlim or ylim is set
    mask = np.ones(len(x), dtype=bool)
    if xlim:
        mask &= (x >= xlim[0]) & (x <= xlim[1])
    if ylim:
        mask &= (y >= ylim[0]) & (y <= ylim[1])
    
    x, y = x[mask], y[mask]
    cell_types = cell_types[mask]

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    # Plot highlighted cell types with larger circles
    if highlight_cell_types is not None:
                # Plot all other cell types in light gray with smaller circles
        for cell_type in color_mapping:
            if cell_type not in highlight_cell_types:
                type_mask = cell_types == cell_type
                ax.scatter(x[type_mask], y[type_mask], color='lightgray', label=cell_type, s=s, alpha=1)
                
        for cell_type in highlight_cell_types:
            type_mask = cell_types == cell_type
            ax.scatter(x[type_mask], y[type_mask], color=color_mapping.get(cell_type, 'black'), 
                        label=cell_type, s=highlight_size, alpha=1, edgecolor='black', linewidth=linewidth)

    else:
        for cell_type in color_mapping:
            type_mask = cell_types == cell_type
            ax.scatter(x[type_mask], y[type_mask], color=color_mapping[cell_type], label=cell_type, s=s, alpha=1)
    
    if show_axes:
        ax.set_title("Cell types")
        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")

    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)

    # Remove top and right spines in the main plot
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if not show_axes:
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)

    # Add legend and show plot
    if show_legend:
        plt.legend(markerscale=5, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize="small", frameon=True)
   
    if savedir is not None:
        if highlight_cell_types is None:
            fig.savefig(savedir / savename / f"full_{obs_key}_lims_x{xlim}_y{ylim}.png", dpi=250, bbox_inches="tight")
        else:
            fig.savefig(savedir / savename / f"highlight{highlight_cell_types}_limsx{xlim}_y{ylim}.png", dpi=250, bbox_inches="tight")
    
    return fig


In [ ]:
def plot_cnmf_scores(adata, score_keys=None, position_key="spatial", 
                      xlim=None, ylim=None, s=2, figsize=(15, 10), 
                      linewidth=0.2, ncols=3, scale_params=None, 
                      show_axes=True, show_colorbar=True,
                      savedir=None, savename=""):
    """
    Plots spatial expression of multiple continuous scores from adata.obs.

    Parameters:
    - adata: AnnData object
    - score_keys: List of keys in adata.obs corresponding to the scores to be plotted.
                  Defaults to ["cNMF_1", "cNMF_2", "cNMF_3", "cNMF_4", "cNMF_5"].
    - position_key: Key to access cell coordinates in adata (default: "spatial").
    - xlim: Tuple (xmin, xmax) to zoom in on a specific x-range (default: None, no zoom).
    - ylim: Tuple (ymin, ymax) to zoom in on a specific y-range (default: None, no zoom).
    - s: Size of the scatter plot points (default: 2).
    - figsize: Size of the overall figure (default: (15, 10)).
    - linewidth: Line width for scatter plot points (default: 0.2).
    - ncols: Number of columns in the subplot grid (default: 3).
    - scale_params: Dictionary where keys are score names and values are dictionaries
                    with optional keys: 'vmin', 'vmax', and 'center' for color scaling.
                    Example: {"cNMF_1": {"vmin": 0, "vmax": 1, "center": 0.5}}.
    - show_axes: Whether to display the x and y axes (default: True).
    - show_colorbar: Whether to show the colorbar (default: True).
    - savedir: Directory to save the figure (default: None, does not save).
    - savename: Base name for the saved figure (default: "").

    Returns:
    - fig: The matplotlib figure object.
    """
    if score_keys is None:
        score_keys = ["cNMF_1", "cNMF_2", "cNMF_3", "cNMF_4", "cNMF_5"]
    
    if scale_params is None:
        scale_params = {}

    # Ensure that all score keys exist in adata.obs
    for key in score_keys:
        if key not in adata.obs:
            raise ValueError(f"Score key '{key}' not found in adata.obs.")

    # Extract spatial positions
    positions = adata.obsm[position_key]
    x, y = positions[:, 0], positions[:, 1]

    # Apply zoom filter if xlim or ylim is set
    mask = np.ones(len(x), dtype=bool)
    if xlim:
        mask &= (x >= xlim[0]) & (x <= xlim[1])
    if ylim:
        mask &= (y >= ylim[0]) & (y <= ylim[1])
    
    x, y = x[mask], y[mask]

    # Determine grid layout
    nrows = int(np.ceil(len(score_keys) / ncols))  # Compute number of rows needed
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)

    for i, key in enumerate(score_keys):
        row, col = divmod(i, ncols)
        ax = axes[row, col]

        # Extract the corresponding score values
        score_values = adata.obs[key].values[mask]

        # Get color scaling parameters
        vmin = scale_params.get(key, {}).get("vmin", np.min(score_values))
        vmax = scale_params.get(key, {}).get("vmax", np.max(score_values))
        center = scale_params.get(key, {}).get("center", None)

        # Use a diverging colormap if center is specified
        cmap = "coolwarm" if center is not None else "viridis"

        # Scatter plot colored by score values
        scatter = ax.scatter(x, y, c=score_values, cmap=cmap, s=s, alpha=0.8, edgecolor="none", linewidth=linewidth,
                             vmin=vmin, vmax=vmax)

        # Add colorbar if enabled
        if show_colorbar:
            cbar = plt.colorbar(scatter, ax=ax, label=key)
            cbar.ax.tick_params(labelsize=8)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        name = key.split("_")
        name = name[0] + "$_" + name[1] + "$"
        ax.set_title(f"{name}", fontsize=25)
        
        if show_axes:
            ax.set_xlabel("X Coordinate")
            ax.set_ylabel("Y Coordinate")
        else:
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)
            ax.spines['bottom'].set_visible(False)
            ax.spines['left'].set_visible(False)

        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)

    # Remove any unused subplots if the number of scores is less than nrows * ncols
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        fig.delaxes(axes[row, col])

    plt.tight_layout()

    if savedir is not None:
        fig.savefig(savedir / savename / f"{savename}_cnmf_scores_x{xlim}_y{ylim}.png", dpi=250, bbox_inches="tight")

    return fig


In [ ]:
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_gene_expression_highlight(adata, gene, obs_key="Cell_type", position_key="spatial", 
                                    highlight_cell_types=None, xlim=None, ylim=None, 
                                    color_mapping=None, s=2, highlight_size=50, figsize=(12, 10), 
                                    linewidth=0.5, show_axes=True, show_colorbar=True,
                                    savedir=None, savename=""):
    """
    Plots the expression of a specific gene and highlights certain cell types.
    The edge color is associated with the cell type, and the hue of the cells is based on gene expression levels.
    Adds a violin plot inset comparing gene expression in the highlighted populations.
    
    Additional Parameters:
    - show_axes: Whether to display the x and y axes (default: True).
    - show_colorbar: Whether to show the colorbar (default: True).
    """
    if gene not in adata.var_names:
        if gene not in adata.obs:
            raise ValueError(f"Gene '{gene}' not found in adata.var_names or in adata.obs.")
        else:
            gene_expression = adata.obs[gene].ravel()
    else:
        gene_idx = adata.var_names.get_loc(gene)
        gene_expression = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, gene_idx]
    
    cell_types = adata.obs[obs_key]
    if color_mapping is None:
        unique_cell_types = np.sort(cell_types.unique())
        palette = sns.color_palette("tab20", len(unique_cell_types)) if len(unique_cell_types) <= 20 else sns.color_palette("husl", len(unique_cell_types))
        color_mapping = {cell_type: palette[i] for i, cell_type in enumerate(unique_cell_types)}
    
    positions = adata.obsm[position_key]
    x, y = positions[:, 0], positions[:, 1]
    
    mask = np.ones(len(x), dtype=bool)
    if xlim:
        mask &= (x >= xlim[0]) & (x <= xlim[1])
    if ylim:
        mask &= (y >= ylim[0]) & (y <= ylim[1])
    
    x, y = x[mask], y[mask]
    gene_expression = gene_expression[mask]
    cell_types = cell_types[mask]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    for cell_type in color_mapping:
        if highlight_cell_types is None or cell_type not in highlight_cell_types:
            type_mask = cell_types == cell_type
            ax.scatter(x[type_mask], y[type_mask], c=gene_expression[type_mask], cmap="viridis", 
                       label=cell_type, s=s, alpha=0.5, edgecolor='none', linewidth=0)
    
    if highlight_cell_types is not None:
        for cell_type in highlight_cell_types:
            type_mask = cell_types == cell_type
            scatter = ax.scatter(x[type_mask], y[type_mask], c=gene_expression[type_mask], cmap="viridis", 
                                 label=cell_type, s=highlight_size, alpha=0.8, edgecolor=color_mapping.get(cell_type, 'black'), 
                                 linewidth=linewidth)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    if show_colorbar:
        cbar = plt.colorbar(scatter, ax=ax, label=f"{gene} Expression")
        cbar.ax.tick_params(labelsize=8)
    
    if show_axes:
        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")
        ax.set_title(f"Expression of {gene}")
    else:
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
    
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    
    if highlight_cell_types is not None:
        inset_ax = inset_axes(ax, width="15%", height="15%", loc="upper right")
        highlighted_data = [gene_expression[cell_types == cell_type] for cell_type in highlight_cell_types]
        other_data = [gene_expression[cell_types != cell_type] for cell_type in highlight_cell_types]
        highlighted_data.append(np.concatenate(other_data))
        combined_data = highlighted_data
        combined_labels = highlight_cell_types + ["Other"]
        sns.violinplot(data=combined_data, ax=inset_ax, inner="stick")
        inset_ax.set_title(f"{gene} expr.", fontsize=10)
        inset_ax.set_ylabel("")
        inset_ax.spines['top'].set_visible(False)
        inset_ax.spines['right'].set_visible(False)
        inset_ax.set_xticks(np.arange(len(combined_labels)))
        inset_ax.set_xticklabels(combined_labels, rotation=45, ha='right', fontsize=8)
    
    if savedir is not None:
        fig.savefig(savedir / savename / f"expr{gene}_highlight{highlight_cell_types}_x{xlim}_y{ylim}.png", dpi=250, bbox_inches="tight")
    return fig

In [ ]:
def plot_gene_expression_inset(adata, gene, obs_key="Cell_type", highlight_cell_types=None, color_mapping=None, 
                               figsize=(4, 3), savedir=None, savename="", xlim=None, ylim=None,):
    """
    Plots a standalone violin plot for gene expression in highlighted and other cell types.

    Parameters:
    - adata: AnnData
        AnnData object containing gene expression and cell type data.
    - gene: str
        Name of the gene to plot.
    - obs_key: str
        Key in adata.obs containing cell type information (default: 'Cell_type').
    - highlight_cell_types: list
        List of cell types to highlight.
    - color_mapping: dict
        Dictionary mapping cell types to colors.
        Example: {'T cells': '#1f77b4', 'B cells': '#ff7f0e'}
    - figsize: tuple
        Size of the figure (default: (4, 3)).

    Returns:
    - fig: matplotlib.figure.Figure
        The matplotlib figure object.
    """
    if gene not in adata.var_names and gene not in adata.obs:
        raise ValueError(f"Gene '{gene}' not found in adata.var_names or adata.obs.")
    
    # Get gene expression values
    if gene in adata.var_names:
        gene_idx = adata.var_names.get_loc(gene)
        gene_expression = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, gene_idx]
    else:
        gene_expression = adata.obs[gene].values
    
    # Get cell types
    cell_types = adata.obs[obs_key]

    # Apply xlim and ylim mask if provided
    mask = np.ones(len(gene_expression), dtype=bool)
    if xlim:
        mask &= (adata.obsm['spatial'][:, 0] >= xlim[0]) & (adata.obsm['spatial'][:, 0] <= xlim[1])
    if ylim:
        mask &= (adata.obsm['spatial'][:, 1] >= ylim[0]) & (adata.obsm['spatial'][:, 1] <= ylim[1])
    
    gene_expression = gene_expression[mask]
    cell_types = cell_types[mask]
    
    # Prepare the data for plotting
    data = []
    labels = []
    colors = []
    
    if highlight_cell_types is not None:
        for cell_type in highlight_cell_types:
            values = gene_expression[cell_types == cell_type]
            data.append(values)
            labels.append(cell_type)
            # Use color mapping if available, otherwise default to grey
            color = color_mapping.get(cell_type, '#808080')
            colors.append(color)
    
    # Add "Other" cells (cells not in highlighted types)
    other_values = gene_expression[~cell_types.isin(highlight_cell_types)]
    data.append(other_values)
    labels.append('Other')
    colors.append('white')  # White fill with black edge
    
    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)
    sns.set(style="white")  # Clean seaborn style
    
    # Plot violin plot
    parts = sns.violinplot(
        data=data,
        ax=ax,
        inner=None,  # Remove inner lines
        linewidth=1
    )
    
    # Set colors manually
    for i, pc in enumerate(parts.collections):
        if i < len(colors):
            pc.set_facecolor(colors[i])
            pc.set_edgecolor('black')

    # Customize plot appearance
    ax.set_title(f"{gene} expr.", fontsize=10)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Remove grid
    ax.grid(False)
    
    if savedir is not None:
        fig.savefig(savedir / savename / f"INSET_expr{gene}_highlight{highlight_cell_types}_x{xlim}_y{ylim}.svg", dpi=250, bbox_inches="tight")
    
    return fig


In [ ]:
def subset_adata_by_region(adata, xlim=None, ylim=None, position_key="spatial"):
    """
    Subsets an AnnData object based on spatial coordinates.

    Parameters:
    - adata: AnnData object
    - xlim: Tuple (xmin, xmax) to filter cells within this x-range (default: None, no filtering)
    - ylim: Tuple (ymin, ymax) to filter cells within this y-range (default: None, no filtering)
    - position_key: Key in adata.obsm that stores spatial coordinates (default: "spatial")

    Returns:
    - subsetted_adata: AnnData object containing only cells within the specified region.
    """
    # Extract spatial coordinates
    positions = adata.obsm[position_key]
    x, y = positions[:, 0], positions[:, 1]

    # Create mask to filter cells
    mask = np.ones(len(x), dtype=bool)
    if xlim:
        mask &= (x >= xlim[0]) & (x <= xlim[1])
    if ylim:
        mask &= (y >= ylim[0]) & (y <= ylim[1])

    # Subset the AnnData object
    subsetted_adata = adata[mask].copy()

    return subsetted_adata


# Run function

In [ ]:
def process_and_plot_data(
    adata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping, 
    figdir, savename, colormapping_cellcharter=None, highlight_size=5, s1=2, s2=1, method_ct_prop="knn", n_neighbors=50, radius=50,
):
    # Compute mean expression scores for specific gene sets
    adata.obs["MDK receiver"] = np.array(adata[:, adata.var_names.intersection(mdk_genes)].X.mean(axis=1)).ravel()
    adata.obs["MIF receiver"] = np.array(adata[:, adata.var_names.intersection(mif_genes)].X.mean(axis=1)).ravel()
    adata.obs["THY1 receiver"] = np.array(adata[:, adata.var_names.intersection(thy1_genes)].X.mean(axis=1)).ravel()
    
    
    # Generate plots
    plot_predicted_cell_types(adata, color_mapping=colormapping, s=s1, figsize=(6, 6), savedir=figdir, savename=savename, show_legend=False, show_axes=False)
    plot_predicted_cell_types(adata, color_mapping=colormapping_cellcharter, obs_key="cluster_cellcharter", s=s1, figsize=(6, 6), savedir=figdir, savename=savename, show_legend=False, show_axes=False)
    plot_predicted_cell_types(adata, color_mapping=colormapping, s=s2, figsize=(6, 6), highlight_cell_types=["Carcinoma", "Inflammatory CAF", "TAM1"], highlight_size=highlight_size, linewidth=0.5, savedir=figdir, savename=savename, show_axes=False, show_legend=False)
    plot_predicted_cell_types(adata, color_mapping=colormapping, s=s2, figsize=(6, 6), highlight_cell_types=["Inflammatory CAF", "TAM1","DC"], highlight_size=highlight_size, linewidth=0.5, savedir=figdir, savename=savename, show_axes=False, show_legend=False)
    plot_predicted_cell_types(adata, color_mapping=colormapping, s=s2, figsize=(6, 6), highlight_cell_types=["Carcinoma", "TAM1", "DC"], highlight_size=highlight_size, linewidth=0.5, savedir=figdir, savename=savename, show_axes=False, show_legend=False)

    mpl.rcParams['font.family'] = 'DejaVu Sans'
    highlight_cell_types = ["Inflammatory CAF", "TAM1", "DC"]
    highlight_cell_types = list(np.intersect1d(highlight_cell_types, adata.obs.Cell_type.unique()))
    for gene in ["THY1", "THY1 receiver"]:
        plot_gene_expression_highlight(adata, gene=gene, obs_key="Cell_type", position_key="spatial", color_mapping=colormapping, highlight_cell_types=highlight_cell_types, s=s2, highlight_size=highlight_size, linewidth=0, figsize=(6, 6), savedir=figdir, savename=savename, show_axes=False, show_colorbar=False)
        
        fig = plot_gene_expression_inset(
            subadata,
            gene=gene,
            obs_key='Cell_type',
            highlight_cell_types = highlight_cell_types,
            color_mapping={ct: colormapping[ct][0] for ct in colormapping},
            figsize=(2,1.5), savedir=figdir, savename=savename,
        )
        
    highlight_cell_types = ["Carcinoma", "Inflammatory CAF", "TAM1"]
    highlight_cell_types = list(np.intersect1d(highlight_cell_types, adata.obs.Cell_type.unique()))
    for gene in ["MDK receiver"]:
        plot_gene_expression_highlight(adata, gene=gene, obs_key="Cell_type", position_key="spatial", color_mapping=colormapping, highlight_cell_types=highlight_cell_types, s=s2, highlight_size=highlight_size, linewidth=0, figsize=(6, 6), savedir=figdir, savename=savename, show_axes=False, show_colorbar=False)
        fig = plot_gene_expression_inset(
            subadata,
            gene=gene,
            obs_key='Cell_type',
            highlight_cell_types = highlight_cell_types,
            color_mapping={ct: colormapping[ct][0] for ct in colormapping},
            figsize=(2,1.5), savedir=figdir, savename=savename,
        )
    highlight_cell_types = ["Carcinoma", "TAM1", "DC"]
    highlight_cell_types = list(np.intersect1d(highlight_cell_types, adata.obs.Cell_type.unique()))
    for gene in ["MIF receiver"]:
        plot_gene_expression_highlight(adata, gene=gene, obs_key="Cell_type", position_key="spatial", color_mapping=colormapping, highlight_cell_types=highlight_cell_types, s=s2, highlight_size=highlight_size, linewidth=0, figsize=(6, 6), savedir=figdir, savename=savename, show_axes=False, show_colorbar=False)
        fig = plot_gene_expression_inset(
            subadata,
            gene=gene,
            obs_key='Cell_type',
            highlight_cell_types = highlight_cell_types,
            color_mapping={ct: colormapping[ct][0] for ct in colormapping},
            figsize=(2,1.5), savedir=figdir, savename=savename,
        )
        
    # Plot cNMF scores
    scale_params = {f"cNMF_{i}": {"vmin": -0.75, "vmax": 0.75, "center": 0} for i in range(1, 6)}
    plot_cnmf_scores(adata, score_keys=list(scale_params.keys()), position_key="spatial", s=s1, figsize=(6.5, 10), scale_params=scale_params, linewidth=0.2, ncols=2, savedir=figdir, savename=savename, show_axes=False, show_colorbar=False)
    
    
    # Compute spatial distances
    sopa.spatial.spatial_neighbors(adata, radius=[0, 50])
    spatial_distance = sopa.spatial.cells_to_groups(adata, "Cell_type", key_added_prefix=None, ignore_zeros=False)
    cell_type_to_cell_type = sopa.spatial.mean_distance(adata, "Cell_type", "Cell_type")
    
    # Plot heatmap
    heatmap_kwargs = {"vmax": 5, "cmap": sns.cm.rocket_r, "cbar_kws": {'label': 'Mean hop distance'}}
    fig, ax = plt.subplots(1, 1, figsize=(3, 2.5))
    sns.heatmap(cell_type_to_cell_type, ax=ax, **heatmap_kwargs)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.savefig(figdir / savename / f"{savename}_celltype_to_celltype_heatmap.png", dpi=250, bbox_inches="tight")

# Set the colors

In [ ]:
figdir = pl.Path("/add/path/here/figures/xenium/")

In [ ]:
import matplotlib 
dicts = []
colorlist = palettable.colorbrewer.sequential.Greys_9.mpl_colors
ctlist = ["T","Treg","NK"]
colormapping_lymphoid = {ct: colorlist[i+3] for i,ct in enumerate(ctlist)}
colormapping_lymphoid["B"] = colorlist[8]
dicts.append(colormapping_lymphoid)

colorlist = palettable.colorbrewer.sequential.Greens_9.mpl_colors
ctlist = ["TAM1","DC","Mast","TAM2"]
colormapping_myeloid = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_myeloid)

colorlist = palettable.colorbrewer.sequential.RdPu_9.mpl_colors
ctlist = ["Endothelial"]
colormapping_endoth = {ct: colorlist[2*i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_endoth)

colorlist = palettable.colorbrewer.sequential.YlOrBr_4.mpl_colors
ctlist = ["Muscle"]
colormapping_muscle = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_muscle)

colorlist = palettable.colorbrewer.sequential.Oranges_5.mpl_colors
ctlist = ["Inflammatory CAF", "Adipose CAF", "Fibroblast"]
colormapping_fibro = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_fibro)

colorlist = palettable.colorbrewer.sequential.Blues_7.mpl_colors
ctlist = ["Epithelial","Epithelial (Pre-cancerous)", "Carcinoma"]
colormapping_epi = {ct: colorlist[2*(i+1)] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_epi)

colormapping = {}
for d in dicts:
    for k, v in d.items():  # d.items() in Python 3+
        colormapping.setdefault(k, []).append(v)

colormapping["Nerve/adrenal"] = [matplotlib.colors.to_rgb("pink")]
colormapping["Adipocyte"] = [matplotlib.colors.to_rgb("darkorange")]

#colormapping = {ct: colormapping[ct][0] for ct in colormapping}

In [ ]:
colormapping_cellcharter = {i: sns.color_palette("bright")[i] for i in range(8)}

In [ ]:
sns.color_palette("bright")

### Get cNMF signatures

In [ ]:
cnmf_sig_dir = pl.Path('/add/path/here/cNMF_malignant_genes_new_cosine')

In [ ]:
cnmf_sigs = {}
for f in cnmf_sig_dir.iterdir():
    cnmf_sigs[f.stem] = pd.read_csv(f, index_col=0).head(100).index

# Get CellChat LR 

In [ ]:
cellchat_df = pd.read_csv("/add/path/here/auxiliary_data/cellchat_database.csv",index_col=0)

In [ ]:
mdk_genes = np.hstack(cellchat_df[cellchat_df["0"]=="MDK"]["1"].str.split("_").ravel())
mif_genes = np.hstack(cellchat_df[cellchat_df["0"]=="MIF"]["1"].str.split("_").ravel())
thy1_genes = np.hstack(cellchat_df[cellchat_df["0"]=="THY1"]["1"].str.split("_").ravel())

# Full Dataset

In [ ]:
full_adata = sc.read_h5ad("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")

# P4

In [ ]:
adata =full_adata[full_adata.obs.patient=="P4"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

In [ ]:
fig = plot_predicted_cell_types(adata, color_mapping=colormapping, s=2, figsize=(6, 6))

# P4_A

In [ ]:
savename = "P4_A"

subadata = subset_adata_by_region(adata, xlim=(0,10500), ylim=(2500,10000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P4_B

In [ ]:
savename = "P4_B"

subadata = subset_adata_by_region(adata, xlim=(43000, 49000), ylim=(12000,19000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P8

In [ ]:
adata = full_adata[full_adata.obs.patient=="P8"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

In [ ]:
savename = "P8"

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    adata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=3, s1=1, s2=0.5,
)

#### Interesting lims to add 
xlim=(10000,30000), ylim=(15000,30000)
xlim=(28000,50000), ylim=(10000,28000)

# P10

In [ ]:
adata = full_adata[full_adata.obs.patient=="P10"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

# P10_A

In [ ]:
savename = "P10_A"

subadata = subset_adata_by_region(adata, xlim=(2000,30000), ylim=(9000,26000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P10_B

In [ ]:
savename = "P10_B"

subadata = subset_adata_by_region(adata, xlim=(17000,27000), ylim=(3000,9000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P10_C

In [ ]:
savename = "P10_C"

subadata = subset_adata_by_region(adata, xlim=(34000,46000), ylim=(11000,22500))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P11

In [ ]:
adata = full_adata[full_adata.obs.patient=="P11"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

# P11_A

In [ ]:
savename = "P11_A"

subadata = subset_adata_by_region(adata, xlim=(0,25000), ylim=(3500,17000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P11_B

In [ ]:
savename = "P11_B"

subadata = subset_adata_by_region(adata, xlim=(19000,31000), ylim=(10000,25000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P11_C

In [ ]:
savename = "P11_C"

subadata = subset_adata_by_region(adata, xlim=(27000,45000), ylim=(1000,12000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P12

In [ ]:
adata = full_adata[full_adata.obs.patient=="P12"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

# P12_A

In [ ]:
savename = "P12_A"

subadata = subset_adata_by_region(adata, xlim=(0,10000), ylim=(4000,15000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P12_B

In [ ]:
savename = "P12_B"

subadata = subset_adata_by_region(adata, xlim=(12000,23000), ylim=(1500,12000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P12_C

In [ ]:
savename = "P12_C"

subadata = subset_adata_by_region(adata, xlim=(23000,34000), ylim=(3000,12000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P12_D

In [ ]:
savename = "P12_D"

subadata = subset_adata_by_region(adata, xlim=(36000,47000), ylim=(3000,11000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P13

In [ ]:
adata = full_adata[full_adata.obs.patient=="P13"].copy()

In [ ]:
adata.var_names = adata.var_names.str.upper()

# P13_A

In [ ]:
savename = "P13_A"

subadata = subset_adata_by_region(adata, xlim=(4000,12000), ylim=(4500,12000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P13_B

In [ ]:
savename = "P13_B"

subadata = subset_adata_by_region(adata, xlim=(9000,24000), ylim=(5000,16000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)

# P13_C

In [ ]:
savename = "P13_C"

subadata = subset_adata_by_region(adata, xlim=(26000,42000), ylim=(1000,11000))

os.makedirs(figdir / savename, exist_ok=True)

process_and_plot_data(
    subadata, mdk_genes, mif_genes, thy1_genes, cnmf_sigs, colormapping,
    figdir, savename, colormapping_cellcharter=colormapping_cellcharter, highlight_size=5, s1=2, s2=1,
)